In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

# -------------------------------
# 1️⃣ Load your dataset
# -------------------------------

x = pd.read_csv("X_test.csv")

y = pd.read_csv("y_test.csv")
y = y.drop(columns=["CustomerId"])

df = pd.DataFrame(x)
df = x.drop(columns=["CustomerId", "Surname"])
df["Gender"] = df["Gender"].map({"Female": 0, "Male": 1})
df["Geography"] = df["Geography"].astype("category").cat.codes
df["Gender"] = df["Gender"].fillna(df["Gender"].median())
df["Balance"] = x["Balance"] / 100000.0
df["EstimatedSalary"] = x["EstimatedSalary"] / 50000.0
df['CreditScore'] = x['CreditScore'] / 650
# display(df)

df2 = pd.DataFrame(y)
# display(df2.head())
# -------------------------------
# 2️⃣ Split features and target
# -------------------------------
X = df.values.astype(np.float32)
y = df2.values.astype(np.int64)

X_tensor = torch.tensor(X)
y_tensor = torch.tensor(y).squeeze()

# print("X shape:", X_tensor.shape)
# print("y shape:", y_tensor.shape)
# print("\nAny NaN in X_tensor? ", torch.isnan(X_tensor).any().item())
# print("Any Inf in X_tensor? ", torch.isinf(X_tensor).any().item())

# print("\nAny NaN in y_tensor? ", torch.isnan(y_tensor).any().item())
# print("Any Inf in y_tensor? ", torch.isinf(y_tensor).any().item())

# print("\nUnique target values:", torch.unique(y_tensor))
print(df.isna().sum())


CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
dtype: int64


In [22]:
# -------------------------------
# 3️⃣ Define the model (10-8-2)
# -------------------------------
class BinaryANN(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=8, output_dim=2):
        super(BinaryANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x  # raw logits

model = BinaryANN()

# -------------------------------
# 4️⃣ Loss and Optimizer
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -------------------------------
# 5️⃣ Train the Model
# -------------------------------
epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 20 == 0:
        _, predicted = torch.max(outputs, 1)
        acc = (predicted == y_tensor).float().mean()
        print(f"Epoch [{epoch+1}/{epochs}] Loss: {loss.item():.4f} | Accuracy: {acc:.2f}")

# -------------------------------
# 6️⃣ Test on a new customer
# -------------------------------
new_customer = torch.tensor([[1.0, 0, 1.0, 45, 5, 0.8, 2, 1, 1, 1.5]], dtype=torch.float32)
with torch.no_grad():
    output = model(new_customer)
    probs = torch.softmax(output, dim=1)
    pred_class = torch.argmax(probs).item()

print("\nPrediction for new customer:")
print("Probabilities:", probs)
print("Predicted Class:", "True (1)" if pred_class == 1 else "False (0)")


Epoch [20/200] Loss: 0.5393 | Accuracy: 0.79
Epoch [40/200] Loss: 0.5381 | Accuracy: 0.80
Epoch [60/200] Loss: 0.5297 | Accuracy: 0.79
Epoch [80/200] Loss: 0.5277 | Accuracy: 0.80
Epoch [100/200] Loss: 0.5254 | Accuracy: 0.80
Epoch [120/200] Loss: 0.5230 | Accuracy: 0.80
Epoch [140/200] Loss: 0.5206 | Accuracy: 0.80
Epoch [160/200] Loss: 0.5182 | Accuracy: 0.80
Epoch [180/200] Loss: 0.5157 | Accuracy: 0.80
Epoch [200/200] Loss: 0.5133 | Accuracy: 0.80

Prediction for new customer:
Probabilities: tensor([[0.8344, 0.1656]])
Predicted Class: False (0)
